# ADF-Test: Schritt für Schritt

## Worum geht es?

Viele Verfahren der Zeitreihenanalyse (z. B. ARIMA) setzen voraus, dass die Daten **stationär** sind – also um einen festen Mittelwert schwanken und sich statistisch über die Zeit nicht verändern.

Der **ADF-Test** (Augmented Dickey-Fuller) prüft genau das: *Ist meine Zeitreihe stationär oder nicht?*

### Die wichtigsten Begriffe

| Begriff | Bedeutung |
|---------|-----------|
| **Stationär** | Die Reihe schwankt um einen festen Mittelwert und kehrt dorthin zurück |
| **Einheitswurzel** | Die Reihe wandert – Schocks häufen sich an (= nicht stationär) |
| **H₀** | Annahme: Einheitswurzel vorhanden → **nicht stationär** |
| **H₁** | Gegenbehauptung: **stationär** |
| **p-Wert** | Wahrscheinlichkeit, die Daten zu sehen, falls H₀ wahr wäre |
| **Δx** | Veränderung: Δx = x_t – x_{t-1} (= x' im Leitfaden) |

> **Achtung – umgekehrte Logik!** H₀ ist die "schlechte Nachricht" (nicht stationär).
> Ein kleiner p-Wert ist ein **gutes** Ergebnis!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 120

## Schritt 1: Stationär vs. Random Walk – visuell erkennen

Wir simulieren zwei Zeitreihen mit derselben Grundformel (vgl. AR(1), Leitfaden Block 6):

**x_t = φ · x_{t-1} + Rauschen**

- **φ = 0,5** → Reihe wird immer wieder zur Mitte gezogen → **stationär**
- **φ = 1,0** → Reihe erbt den vorigen Wert komplett → **wandert** (Random Walk)

In [ ]:
np.random.seed(42)
n = 200
eps = np.random.normal(0, 1, n)

# ── Stationäre Reihe: φ = 0.5 ──
x_stat = np.zeros(n)
for t in range(1, n):
    x_stat[t] = 0.5 * x_stat[t-1] + eps[t]

# ── Random Walk: φ = 1.0 ──
x_rw = np.zeros(n)
for t in range(1, n):
    x_rw[t] = 1.0 * x_rw[t-1] + eps[t]

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(x_stat, color='#2E75B6', linewidth=1.2)
axes[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[0].set_title('Stationär (φ = 0.5) – kehrt immer zurück')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x_t')

axes[1].plot(x_rw, color='#C0392B', linewidth=1.2)
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_title('Random Walk (φ = 1.0) – wandert weg')
axes[1].set_xlabel('t')
axes[1].set_ylabel('x_t')

plt.tight_layout()
plt.show()

print("Stationäre Reihe:  Mittelwert =", round(x_stat.mean(), 2), " Std.abw. =", round(x_stat.std(), 2))
print("Random Walk:       Mittelwert =", round(x_rw.mean(), 2), " Std.abw. =", round(x_rw.std(), 2))

## Schritt 2: Den ADF-Test durchführen

Jetzt lassen wir den ADF-Test entscheiden. Die Funktion `adfuller()` aus `statsmodels` berechnet:
- eine **Teststatistik** (wie stark die Evidenz gegen H₀ ist)
- einen **p-Wert** (unter der speziellen Dickey-Fuller-Verteilung)

**Entscheidungsregel:**
- p-Wert **< 0,05** → H₀ verwerfen → Reihe ist **stationär** ✅
- p-Wert **≥ 0,05** → H₀ nicht verworfen → **Einheitswurzel** nicht auszuschließen ❌

In [ ]:
def adf_test(x, label):
    """ADF-Test durchführen und Ergebnis klar ausgeben."""
    ergebnis = adfuller(x, regression='c')
    adf_statistik = ergebnis[0]
    p_wert        = ergebnis[1]
    krit_werte    = ergebnis[4]
    
    print(f"── {label} ──")
    print(f"   ADF-Statistik:  {adf_statistik:.4f}")
    print(f"   p-Wert:         {p_wert:.4f}")
    print(f"   Kritische Werte:")
    for niveau, wert in krit_werte.items():
        marker = "  ← verworfen" if adf_statistik < wert else ""
        print(f"     {niveau}: {wert:.4f}{marker}")
    
    if p_wert < 0.05:
        print(f"\n   ✅ p = {p_wert:.4f} < 0.05 → H₀ verworfen → STATIONÄR")
    else:
        print(f"\n   ❌ p = {p_wert:.4f} ≥ 0.05 → H₀ nicht verworfen → EINHEITSWURZEL")
    print()

adf_test(x_stat, "Stationäre Reihe (φ = 0.5)")
adf_test(x_rw,   "Random Walk (φ = 1.0)")

## Schritt 3: Wozu differenzieren?

Wenn der ADF-Test "Einheitswurzel" sagt, **differenziert** man die Reihe (vgl. Leitfaden Block 5):
statt der Originalwerte betrachtet man die **Veränderung** von Zeitschritt zu Zeitschritt.

### Praxisbeispiel: Monatlicher Umsatz

Eine Firma, deren Umsatz:
- insgesamt **steigt** (Trend)
- im **Dezember** immer einen Sprung macht (Saisonalität)
- aber auch zufällig schwankt

Die Umsatzzahlen selbst wandern → nicht stationär.
Aber die **monatlichen Veränderungen** zeigen ein klares Muster → stationär und modellierbar!

> **Merksatz:** Differenzieren entfernt das wandernde Niveau und legt die modellierbaren Muster frei.

In [ ]:
# ── Realistisches Beispiel: Umsatz mit Trend + Saisonalität ──
np.random.seed(7)
monate = 60  # 5 Jahre

trend = np.linspace(0, 30, monate)
saison = 5 * np.sin(2 * np.pi * np.arange(monate) / 12)
rauschen = np.random.normal(0, 2, monate)

umsatz = 100 + trend + saison + np.cumsum(rauschen * 0.5)

# Differenzieren (Leitfaden Block 5: x'_t = x_t − x_{t-1})
delta_umsatz = np.diff(umsatz)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(umsatz, color='#2E75B6', linewidth=1.5)
axes[0].set_title('Monatlicher Umsatz (nicht stationär)')
axes[0].set_xlabel('Monat')
axes[0].set_ylabel('Umsatz (Tsd. €)')

axes[1].plot(delta_umsatz, color='#27AE60', linewidth=1.2)
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_title('Monatliche Veränderung Δx (stationär, mit Muster!)')
axes[1].set_xlabel('Monat')
axes[1].set_ylabel('Veränderung (Tsd. €)')

plt.tight_layout()
plt.show()

# ADF-Test auf beiden
print("VOR dem Differenzieren:")
adf_test(umsatz, "Umsatz (Original)")

print("NACH dem Differenzieren:")
adf_test(delta_umsatz, "Umsatz (Differenziert)")

### Wann differenzieren? – Entscheidungstabelle

| Situation | Was tun? |
|-----------|----------|
| ADF-Test sagt "stationär" | Direkt modellieren, nicht differenzieren |
| ADF-Test sagt "Einheitswurzel" | Differenzieren (Δx bilden), dann erneut testen |
| Differenzierte Reihe ist stationär | Auf Δx modellieren → ARIMA-Modell (Leitfaden Block 9) |
| Auch nach Differenzierung nicht stationär | Nochmals differenzieren (selten nötig) |

**Der Parameter d in ARIMA(p,d,q)** gibt an, wie oft differenziert wurde. Der ADF-Test sagt Ihnen, ob und wie oft dieser Schritt nötig ist.

## Zum Experimentieren

Ändern Sie `phi` im Code unten und beobachten Sie:
- Bei welchem φ kippt der ADF-Test?
- Wie sieht die Reihe bei φ = 0,95 aus – stationär oder nicht?
- Was passiert bei φ = −0,5?

In [ ]:
# ── Hier φ ändern und Zelle ausführen ──
phi = 0.95

np.random.seed(99)
x_exp = np.zeros(200)
for t in range(1, 200):
    x_exp[t] = phi * x_exp[t-1] + np.random.normal(0, 1)

plt.figure(figsize=(12, 4))
plt.plot(x_exp, color='#2E75B6', linewidth=1.2)
plt.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.title(f'AR(1) mit φ = {phi}')
plt.xlabel('t')
plt.ylabel('x_t')
plt.show()

print()
adf_test(x_exp, f"Experiment mit φ = {phi}")